In [ ]:
from hda import Client, Configuration
import logging
import glob
import os
from pathlib import Path

/Users/johannesgille/Desktop/CE_2026_4_drought/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Configure credentials and load `hda` Client

In [ ]:

hdarc = Path(Path.home()/'.hdarc')
if not hdarc.is_file():
    import getpass
    USERNAME = input('Enter your username: ')
    PASSWORD = getpass.getpass('Enter your password: ')

    with open(Path.home()/'.hdarc', 'w') as f:
        f.write(f'user: {USERNAME}\n')
        f.write(f'password:{PASSWORD}\n')
else:
    print('Configuration file already exists.')
    
hda_client = Client()

Configuration file already exists.


## Create the request

In [6]:
dataset_names = [
    "Crop Types",
    "Main Crop Emergence",
    "Main Crop Duration"
]

In [7]:
query = {
  "dataset_id": "EO:EEA:DAT:HRL:CRL",
  "bbox": [
    5.145313352924855,
    47.0331291590884,
    15.986751684828768,
    55.12720853099112
  ],
  "productType": "Crop Types",
  "resolution": "10m",
  "year": "2023",
  "startIndex": 0
}

## Search data

In [ ]:
for dataset in dataset_names:
    print("processing dataset: ", dataset)
    query["productType"] = dataset
    matches = hda_client.search(query)
    print(matches)
    
    OUTPUT_PATH = os.path.join("data", dataset.replace(" ", "_"))
    
    matches.download(OUTPUT_PATH)
    

processing dataset:  Crop Types
SearchResults[items=85,volume=554.5MB]


In [ ]:
for dataset in dataset_names:
    print("processing dataset: ", dataset)
    
    OUTPUT_PATH = os.path.join("data", dataset.replace(" ", "_"))
    for f in glob.glob(os.path.join(OUTPUT_PATH, "*.zip")):
        new_dir = f.replace(".zip", "")
        if not os.path.isdir():
            os.system(f'unzip  -n "{f}" -d "{new_dir}"')

processing dataset:  Crop Types
Archive:  data/Crop_Types/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00.zip
  inflating: data/Crop_Types/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00.tif  
  inflating: data/Crop_Types/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00.xml  
  inflating: data/Crop_Types/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00.tif.aux.xml  
  inflating: data/Crop_Types/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00/CLMS_HRLVLCC_CTY_R10.clr  
  inflating: data/Crop_Types/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00/CLMS_HRLVLCC_CTY_R10.lyr  
  inflating: data/Crop_Types/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00/CLMS_HRLVLCC_CTY_R10.qml  
  inflating: data/Crop_Types/CLMS_HRLVLCC_CTY_S2023_R10m_E43N33_03035_V01_R00/CLMS_HRLVLCC_CTY_R10.sld  
Archive:  data/Crop_Types/CLMS_HRLVLCC_CTY_S2023_R10m_E43N34_0

In [7]:
len(glob.glob( "crop_map_data/*"))

850

In [5]:
OUTPUT_PATH = 'crop_map_data_zipped'

files = [e.split(".")[0].split("/")[1] for e in glob.glob(OUTPUT_PATH + "/*.zip")]

In [6]:
matches_filtered = [match for match in matches.results if (match["properties"]["location"].split(".")[0] not in files)]

In [7]:
len(matches_filtered)

0

In [ ]:
matches.results = matches_filtered

## Download file(s)

On WEkEO's JupyterHub you are limited to 20GB of stockage space, so be careful of the total size of files your request generated.  

### Download files in the current working directory

You can run `matches.download()` to download all the files of your request.  
Please [read the documentation](https://hda.readthedocs.io/en/latest/usage.html#advanced-client-usage) for advanced usage such as:
- downloading first result: `matches[0].download()`
- downloading last result: `matches[-1].download()`
- downloading first 10 results: `matches[:10].download()`
- downloading even results: `matches[::2].download()`
- etc.

For the purpose of this example, we are going to fetch the last result:

In [ ]:
matches.download(OUTPUT_PATH)

In [ ]:
!gdalbuildvrt -srcnodata 0 -vrtnodata 0 crop_map_mosaic.vrt crop_map_data/**/*.tif